<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end passwordless AWS authentication, live agent invocation, and observability trace extraction for recruiters. It uses AWS SigV4 + IAM credentials from the Colab runtime and retrieves observability secrets from AWS SSM securely.


In [ ]:
# Install dependencies silently
!pip install -q rpy2
!sudo apt-get update -y -q > /dev/null && sudo apt-get install -y -q r-base neofetch > /dev/null
!R -q -e "install.packages(c('jsonlite','httr','reticulate'), repos='https://cloud.r-project.org')" > /dev/null
!neofetch

import requests
from pathlib import Path
from rpy2 import robjects

R_RAW = "https://raw.githubusercontent.com/joseguzman1337/aws-financial-ai-agent/main/r/notebook_runtime.R"
R_FILE = Path('/tmp/notebook_runtime.R')
R_FILE.write_text(requests.get(R_RAW, timeout=30).text, encoding='utf-8')

robjects.r(f"source('{R_FILE}')")
robjects.r("rt <- runtime_init(); rt <- refresh_clients(rt)")
print(f"Loaded R runtime from GitHub: {R_RAW}")



### 1. Verify AWS Identity
Confirm the notebook runtime has AWS credentials and can call AWS APIs.


In [ ]:
robjects.r('rt <- ensure_fresh(rt, force=TRUE); cat("Identity refreshed\n")')


### 2. SigV4 Auth Mode
No Cognito username/password is required. Agent calls are signed directly with AWS SigV4.


In [ ]:
print("No Cognito password flow required in notebook.")
print("Mode: GitHub-loaded R runtime.")


### 3. Invoke the Agentcore Streaming Endpoint

In [ ]:
robjects.r('cat("Agent URL: ", agent_url(rt), "\nSession ID: ", rt$session_id, "\n")')


### 4. Execute Required Financial Queries

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
]
for q in queries:
    robjects.globalenv['py_prompt'] = q
    robjects.r('rt <- query_agent(rt, py_prompt)')



### 5. Observability Verification
Fetch Langfuse traces using keys from AWS SSM and verify API connectivity. Session traces may appear with delay.


In [ ]:
robjects.r('rt <- verify_observability(rt)')
